In [1]:
import numpy as np
import matplotlib.pyplot as plt
import importlib

import tides
import pairwise_local_interaction_library as plil

print("TIDES loaded from:")
print(tides.__file__)
print("\nPairwise local-interaction library loaded from:")
print(plil.__file__)


TIDES loaded from:
/home/liu.xuanc/ondemand/TIDES/tides.py

Pairwise local-interaction library loaded from:
/home/liu.xuanc/ondemand/TIDES/pairwise_local_interaction_library.py


In [2]:
# ------------------------------------------------------------
# N=8 benchmark truth: temporal schedule only
# ------------------------------------------------------------

N = 8
dt = 5e-4
T_total = 0.36

true_change_times = np.array([
    0.06,
    0.12,
    0.18,
    0.24,
    0.30,
])

true_transition_indices = np.rint(true_change_times / dt).astype(int)

print("True change times   :", true_change_times)
print("True transition idx :", true_transition_indices)


True change times   : [0.06 0.12 0.18 0.24 0.3 ]
True transition idx : [120 240 360 480 600]


In [3]:
# ============================================================
# Cell 3 — General pairwise N=8 canonical benchmark
# ============================================================
#
# This benchmark is deliberately outside the old D q class.
# Each active edge {i,j} contributes independently to BOTH
# endpoints through a swap-equivariant local law
#
#   g_i = w_ij [theta1 x_j + theta2 x_j^2 + theta3 x_i x_j]
#   g_j = w_ij [theta1 x_i + theta2 x_i^2 + theta3 x_i x_j]
#
# It therefore tests the new endpoint-output representation.
# The law lies exactly in the default degree-2 identifiable
# pairwise polynomial library:
#
#   neighbor, neighbor^2, self*neighbor.
#
# Inference does NOT receive theta, active snapshots, or weights.
# ============================================================

from itertools import combinations
import numpy as np

SEED = 20260811

N = 8
DT = 5.0e-4
STAGE_DURATION = 0.060
N_STAGES = 6
INTERVALS_PER_STAGE = int(round(STAGE_DURATION / DT))

assert INTERVALS_PER_STAGE == 120

# ------------------------------------------------------------
# Shared local interaction law in polynomial-library coordinates
# ------------------------------------------------------------

PAIR_LAW_LABELS_TRUE = (
    "neighbor",
    "neighbor^2",
    "self*neighbor",
)

PAIR_LAW_TRUE = np.array(
    [1.0, 0.18, -0.62],
    dtype=float,
)

# ------------------------------------------------------------
# Complete candidate pair space: all 28 undirected pairs
# ------------------------------------------------------------

CANDIDATE_EDGES = tuple(
    combinations(range(1, N + 1), 2)
)

assert len(CANDIDATE_EDGES) == 28

EDGE_INDEX_1B = {
    edge: m
    for m, edge in enumerate(CANDIDATE_EDGES)
}

# ------------------------------------------------------------
# Fixed heterogeneous microscopic edge weights
# ------------------------------------------------------------

weight_rng = np.random.default_rng(SEED)

EDGE_WEIGHT = {
    edge: int(weight_rng.integers(800, 1201)) / 1000.0
    for edge in CANDIDATE_EDGES
}

# ------------------------------------------------------------
# Six connected snapshots; 4 changed edges per transition
# ------------------------------------------------------------

SNAPSHOTS = (
    (
        (1, 3), (2, 5), (2, 8), (3, 5), (3, 8),
        (4, 6), (4, 7), (5, 6), (6, 8), (7, 8),
    ),
    (
        (1, 3), (1, 5), (2, 3), (2, 8), (3, 5),
        (3, 8), (4, 6), (4, 7), (5, 6), (7, 8),
    ),
    (
        (1, 3), (1, 6), (2, 3), (2, 7), (2, 8),
        (3, 8), (4, 6), (4, 7), (5, 6), (7, 8),
    ),
    (
        (1, 3), (1, 4), (1, 6), (2, 7), (2, 8),
        (3, 8), (4, 5), (4, 6), (5, 6), (7, 8),
    ),
    (
        (1, 3), (1, 6), (2, 7), (2, 8), (3, 8),
        (4, 5), (5, 6), (5, 7), (6, 7), (7, 8),
    ),
    (
        (1, 3), (1, 8), (2, 8), (3, 8), (4, 5),
        (4, 8), (5, 6), (5, 7), (6, 7), (7, 8),
    ),
)

# ------------------------------------------------------------
# Initial condition
# ------------------------------------------------------------

X0 = np.array(
    [
        -0.319989,
        -0.301418,
         0.354093,
        -0.213312,
        -0.026615,
         0.067347,
         0.282008,
         0.157887,
    ],
    dtype=float,
)
X0 -= X0.mean()

# ------------------------------------------------------------
# True edge-local endpoint law
# ------------------------------------------------------------

def edge_field(x, edge):
    i, j = edge
    i -= 1
    j -= 1

    xi = x[i]
    xj = x[j]

    # Same shared law on every active edge; edge-specific strength
    # is carried only by EDGE_WEIGHT.
    gi = (
        PAIR_LAW_TRUE[0] * xj
        + PAIR_LAW_TRUE[1] * xj**2
        + PAIR_LAW_TRUE[2] * xi * xj
    )

    gj = (
        PAIR_LAW_TRUE[0] * xi
        + PAIR_LAW_TRUE[1] * xi**2
        + PAIR_LAW_TRUE[2] * xi * xj
    )

    out = np.zeros_like(x, dtype=float)
    w = EDGE_WEIGHT[edge]
    out[i] += w * gi
    out[j] += w * gj
    return out


def stage_field(x, stage):
    out = np.zeros(N, dtype=float)
    for edge in SNAPSHOTS[stage]:
        out += edge_field(x, edge)
    return out


def rk4_step(x, dt, stage):
    k1 = stage_field(x, stage)
    k2 = stage_field(x + 0.5 * dt * k1, stage)
    k3 = stage_field(x + 0.5 * dt * k2, stage)
    k4 = stage_field(x + dt * k3, stage)
    return x + (dt / 6.0) * (k1 + 2.0*k2 + 2.0*k3 + k4)

# ------------------------------------------------------------
# Structural audits
# ------------------------------------------------------------

def is_connected(edge_set):
    adjacency = {i: set() for i in range(1, N + 1)}
    for i, j in edge_set:
        adjacency[i].add(j)
        adjacency[j].add(i)
    seen = {1}
    stack = [1]
    while stack:
        u = stack.pop()
        for v in adjacency[u]:
            if v not in seen:
                seen.add(v)
                stack.append(v)
    return len(seen) == N


def is_forest(edge_set):
    parent = {i: i for i in range(1, N + 1)}

    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]
            a = parent[a]
        return a

    for a, b in edge_set:
        ra = find(a)
        rb = find(b)
        if ra == rb:
            return False
        parent[ra] = rb
    return True


transition_supports_true = []
for k in range(N_STAGES - 1):
    before = set(SNAPSHOTS[k])
    after = set(SNAPSHOTS[k + 1])
    transition_supports_true.append(tuple(sorted(before ^ after)))

assert all(len(snapshot) == 10 for snapshot in SNAPSHOTS)
assert all(len(set(snapshot)) == len(snapshot) for snapshot in SNAPSHOTS)
assert all(set(snapshot).issubset(CANDIDATE_EDGES) for snapshot in SNAPSHOTS)
assert all(is_connected(snapshot) for snapshot in SNAPSHOTS)
assert all(len(S) == 4 for S in transition_supports_true)
assert all(is_forest(S) for S in transition_supports_true)

true_change_times = np.arange(1, N_STAGES) * STAGE_DURATION
true_transition_indices = np.rint(true_change_times / DT).astype(int)

active_union = set().union(*map(set, SNAPSHOTS))

print("=" * 90)
print("TIDES N=8 general-pairwise canonical benchmark")
print("=" * 90)
print("N                       :", N)
print("candidate edges M       :", len(CANDIDATE_EDGES))
print("active edges / stage    :", [len(G) for G in SNAPSHOTS])
print("unique active edges     :", len(active_union))
print("dt                      :", DT)
print("stage duration          :", STAGE_DURATION)
print("intervals / stage       :", INTERVALS_PER_STAGE)
print("true library labels     :", PAIR_LAW_LABELS_TRUE)
print("true shared theta       :", PAIR_LAW_TRUE)
print("output class            : general endpoint pair (not D q)")

print("\nTransition structural load:")
for k, S in enumerate(transition_supports_true, start=1):
    before = set(SNAPSHOTS[k - 1])
    after = set(SNAPSHOTS[k])
    removed = tuple(sorted(before - after))
    added = tuple(sorted(after - before))
    print(
        f"G{k}->G{k+1}: s={len(S)}, removed={len(removed)}, "
        f"added={len(added)}, forest={is_forest(S)}, changes={S}"
    )

print("\nTrue change times   :", true_change_times)
print("True transition idx :", true_transition_indices)


TIDES N=8 general-pairwise canonical benchmark
N                       : 8
candidate edges M       : 28
active edges / stage    : [10, 10, 10, 10, 10, 10]
unique active edges     : 20
dt                      : 0.0005
stage duration          : 0.06
intervals / stage       : 120
true library labels     : ('neighbor', 'neighbor^2', 'self*neighbor')
true shared theta       : [ 1.    0.18 -0.62]
output class            : general endpoint pair (not D q)

Transition structural load:
G1->G2: s=4, removed=2, added=2, forest=True, changes=((1, 5), (2, 3), (2, 5), (6, 8))
G2->G3: s=4, removed=2, added=2, forest=True, changes=((1, 5), (1, 6), (2, 7), (3, 5))
G3->G4: s=4, removed=2, added=2, forest=True, changes=((1, 4), (2, 3), (4, 5), (4, 7))
G4->G5: s=4, removed=2, added=2, forest=True, changes=((1, 4), (4, 6), (5, 7), (6, 7))
G5->G6: s=4, removed=2, added=2, forest=True, changes=((1, 6), (1, 8), (2, 7), (4, 8))

True change times   : [0.06 0.12 0.18 0.24 0.3 ]
True transition idx : [120 240 360

In [4]:
# ============================================================
# Cell 4 — Generate general-pairwise benchmark trajectory
# ============================================================

n_total_intervals = N_STAGES * INTERVALS_PER_STAGE
T = np.arange(n_total_intervals + 1, dtype=float) * DT

X = np.empty((n_total_intervals + 1, N), dtype=float)
X[0] = X0

true_stage_of_interval = np.repeat(
    np.arange(N_STAGES, dtype=int),
    INTERVALS_PER_STAGE,
)

for n, stage in enumerate(true_stage_of_interval):
    X[n + 1] = rk4_step(X[n], DT, int(stage))

t = T.copy()

state_displacement = np.linalg.norm(X[-1] - X[0])
max_abs_state = np.max(np.abs(X))
net_sum_change = float(X[-1].sum() - X[0].sum())

print("=" * 90)
print("Cell 4 — general-pairwise trajectory generated")
print("=" * 90)
print("state samples          :", len(X))
print("trajectory intervals   :", len(X) - 1)
print("total duration         :", f"{t[-1]:.6f}")
print("X shape                :", X.shape)
print("||X(T)-X(0)||          :", f"{state_displacement:.6e}")
print("max |x_i(t)|           :", f"{max_abs_state:.6e}")
print("sum_i x_i change       :", f"{net_sum_change:.6e}")
print("all finite             :", np.isfinite(X).all())

# The generalized endpoint output is intentionally non-conservative.
assert X.shape == (721, 8)
assert t.shape == (721,)
assert np.allclose(np.diff(t), DT)
assert np.isfinite(X).all()
assert abs(net_sum_change) > 1e-3
assert np.array_equal(
    true_transition_indices,
    np.array([120, 240, 360, 480, 600])
)


Cell 4 — general-pairwise trajectory generated
state samples          : 721
trajectory intervals   : 720
total duration         : 0.360000
X shape                : (721, 8)
||X(T)-X(0)||          : 2.757140e-01
max |x_i(t)|           : 3.540929e-01
sum_i x_i change       : 4.504142e-01
all finite             : True


In [5]:
# ============================================================
# Cell 5 — Formal TIDES Step 1: change-point detection
# ============================================================

# A. Controlled-count regression
s1_controlled = tides.detect_changes(
    X,
    t,
    method="secant",
    n_changes=5,
    min_separation=10,
)

print("=" * 90)
print("TIDES Step 1 — controlled-count regression")
print("=" * 90)
print("true indices       :", true_transition_indices)
print("detected indices   :", s1_controlled.transition_indices)
print("index errors       :", s1_controlled.transition_indices - true_transition_indices)
print("true times         :", true_change_times)
print("detected times     :", s1_controlled.transition_times)

# B. Fully blind robust threshold
V_SECANT = np.diff(X, axis=0) / DT
jump = np.linalg.norm(V_SECANT[1:] - V_SECANT[:-1], axis=1)
jump_median = np.median(jump)
jump_mad = np.median(np.abs(jump - jump_median))
blind_threshold = jump_median + 20.0 * jump_mad

s1_blind = tides.detect_changes(
    X,
    t,
    method="secant",
    threshold=blind_threshold,
    min_separation=1,
)

print("\n" + "=" * 90)
print("TIDES Step 1 — fully blind detection")
print("=" * 90)
print("jump median        :", f"{jump_median:.6e}")
print("jump MAD           :", f"{jump_mad:.6e}")
print("blind threshold    :", f"{blind_threshold:.6e}")
print("detected count     :", s1_blind.n_transitions)
print("detected indices   :", s1_blind.transition_indices)
print("detected times     :", s1_blind.transition_times)

controlled_exact = np.array_equal(
    s1_controlled.transition_indices,
    true_transition_indices,
)
blind_exact = np.array_equal(
    s1_blind.transition_indices,
    true_transition_indices,
)

print("\n" + "=" * 90)
print("controlled exact   :", controlled_exact)
print("blind exact        :", blind_exact)
assert controlled_exact
assert blind_exact
print("TIDES STEP 1: PASS")
print("=" * 90)


TIDES Step 1 — controlled-count regression
true indices       : [120 240 360 480 600]
detected indices   : [120 240 360 480 600]
index errors       : [0 0 0 0 0]
true times         : [0.06 0.12 0.18 0.24 0.3 ]
detected times     : [0.06 0.12 0.18 0.24 0.3 ]

TIDES Step 1 — fully blind detection
jump median        : 7.526400e-04
jump MAD           : 1.037306e-04
blind threshold    : 2.827251e-03
detected count     : 5
detected indices   : [120 240 360 480 600]
detected times     : [0.06 0.12 0.18 0.24 0.3 ]

controlled exact   : True
blind exact        : True
TIDES STEP 1: PASS


In [6]:
# ============================================================
# Cell 6 — PREPROCESSING
#          Step-1 segmentation + 4-point midpoint reconstruction
# ============================================================
# From this point onward segmentation is determined only by
# the BLIND Step-1 output.


detected_transition_indices = s1_blind.transition_indices.copy()
detected_transition_times = s1_blind.transition_times.copy()

V_SECANT = np.diff(X, axis=0) / DT
n_intervals = len(V_SECANT)

segment_bounds = np.concatenate(
    ([0], detected_transition_indices, [n_intervals])
)
segment_slices = tuple(
    slice(int(segment_bounds[k]), int(segment_bounds[k + 1]))
    for k in range(len(segment_bounds) - 1)
)
segment_interval_counts = np.array(
    [sl.stop - sl.start for sl in segment_slices],
    dtype=int,
)

stage_of_interval = np.empty(n_intervals, dtype=int)
for stage, sl in enumerate(segment_slices):
    stage_of_interval[sl] = stage

obs_interval_indices = []
X_MID = []
V_MID = []
OBS_STAGE = []
T_MID = []

for n in range(1, n_intervals - 1):
    same_stage = (
        stage_of_interval[n - 1]
        == stage_of_interval[n]
        == stage_of_interval[n + 1]
    )
    if not same_stage:
        continue

    x_mid = (
        -X[n - 1]
        + 9.0 * X[n]
        + 9.0 * X[n + 1]
        - X[n + 2]
    ) / 16.0

    v_mid = (
        X[n - 1]
        - 27.0 * X[n]
        + 27.0 * X[n + 1]
        - X[n + 2]
    ) / (24.0 * DT)

    obs_interval_indices.append(n)
    X_MID.append(x_mid)
    V_MID.append(v_mid)
    T_MID.append((t[n] + t[n + 1]) / 2.0)
    OBS_STAGE.append(stage_of_interval[n])

obs_interval_indices = np.asarray(obs_interval_indices, dtype=int)
X_MID = np.asarray(X_MID, dtype=float)
V_MID = np.asarray(V_MID, dtype=float)
T_MID = np.asarray(T_MID, dtype=float)
OBS_STAGE = np.asarray(OBS_STAGE, dtype=int)

TIDES_OBS = {
    "t_state": t.copy(),
    "x_state": X.copy(),
    "detected_transition_indices": detected_transition_indices.copy(),
    "detected_transition_times": detected_transition_times.copy(),
    "segment_bounds": segment_bounds.copy(),
    "segment_interval_counts": segment_interval_counts.copy(),
    "obs_interval_indices": obs_interval_indices.copy(),
    "t_mid": T_MID.copy(),
    "x_mid": X_MID.copy(),
    "velocity_mid": V_MID.copy(),
    "stage_of_observation": OBS_STAGE.copy(),
}

samples_per_stage = np.bincount(OBS_STAGE, minlength=N_STAGES)

print("=" * 90)
print("Cell 6 — PREPROCESSING")
print("=" * 90)
print("detected transitions   :", detected_transition_indices)
print("segment lengths        :", segment_interval_counts.tolist())
print("inference observations :", len(X_MID))
print("samples / stage        :", samples_per_stage.tolist())
print("X_MID shape            :", X_MID.shape)
print("V_MID shape            :", V_MID.shape)
print("OBS_STAGE shape        :", OBS_STAGE.shape)

assert X_MID.shape == (708, 8)
assert V_MID.shape == (708, 8)
assert OBS_STAGE.shape == (708,)
assert np.array_equal(samples_per_stage, np.array([118] * 6))


Cell 6 — PREPROCESSING
detected transitions   : [120 240 360 480 600]
segment lengths        : [120, 120, 120, 120, 120, 120]
inference observations : 708
samples / stage        : [118, 118, 118, 118, 118, 118]
X_MID shape            : (708, 8)
V_MID shape            : (708, 8)
OBS_STAGE shape        : (708,)


In [7]:
# ============================================================
# Cell 7 — PREPROCESSING DIAGNOSTIC
#          4-point vs 6-point numerical-resolution audit
# ============================================================

import math


def finite_difference_weights(nodes, derivative_order):
    nodes = np.asarray(nodes, dtype=float)
    n = len(nodes)
    A = np.vstack([nodes**k for k in range(n)])
    b = np.zeros(n, dtype=float)
    b[derivative_order] = math.factorial(derivative_order)
    return np.linalg.solve(A, b)


z4 = np.array([-1.5, -0.5, 0.5, 1.5], dtype=float)
z6 = np.array([-2.5, -1.5, -0.5, 0.5, 1.5, 2.5], dtype=float)

w4_x = finite_difference_weights(z4, 0)
w4_d = finite_difference_weights(z4, 1)
w6_x = finite_difference_weights(z6, 0)
w6_d = finite_difference_weights(z6, 1)

rel_state_46 = []
rel_velocity_46 = []

for n in range(2, n_intervals - 2):
    stencil_stages = {
        stage_of_interval[n + j]
        for j in (-2, -1, 0, 1, 2)
    }
    if len(stencil_stages) != 1:
        continue

    p4 = X[n - 1 : n + 3]
    p6 = X[n - 2 : n + 4]

    x4 = (w4_x[:, None] * p4).sum(axis=0)
    x6 = (w6_x[:, None] * p6).sum(axis=0)
    v4 = (w4_d[:, None] * p4).sum(axis=0) / DT
    v6 = (w6_d[:, None] * p6).sum(axis=0) / DT

    rel_state_46.append(
        np.linalg.norm(x6 - x4) / max(np.linalg.norm(x6), 1e-15)
    )
    rel_velocity_46.append(
        np.linalg.norm(v6 - v4) / max(np.linalg.norm(v6), 1e-15)
    )

rel_state_46 = np.asarray(rel_state_46)
rel_velocity_46 = np.asarray(rel_velocity_46)

resolution_rel_max = max(
    float(rel_state_46.max()),
    float(rel_velocity_46.max()),
    1e-14,
)

# Conservative externally calibrated uncertainty floor.
UNCERTAINTY_FLOOR = 20.0 * resolution_rel_max

TIDES_OBS.update({
    "rel_state_4v6": rel_state_46.copy(),
    "rel_velocity_4v6": rel_velocity_46.copy(),
    "resolution_rel_max": resolution_rel_max,
    "uncertainty_floor": UNCERTAINTY_FLOOR,
})

print("=" * 90)
print("Cell 7 — PREPROCESSING DIAGNOSTIC")
print("=" * 90)
print("resolution samples       :", len(rel_state_46))
print("max relative state 4-vs-6:", f"{rel_state_46.max():.6e}")
print("max relative vel. 4-vs-6 :", f"{rel_velocity_46.max():.6e}")
print("median relative velocity :", f"{np.median(rel_velocity_46):.6e}")
print("resolution relative max  :", f"{resolution_rel_max:.6e}")
print("uncertainty floor        :", f"{UNCERTAINTY_FLOOR:.6e}")

assert np.isfinite(UNCERTAINTY_FLOOR)
assert UNCERTAINTY_FLOOR > 0.0


Cell 7 — PREPROCESSING DIAGNOSTIC
resolution samples       : 696
max relative state 4-vs-6: 3.773238e-14
max relative vel. 4-vs-6 : 9.828955e-13
median relative velocity : 5.302760e-13
resolution relative max  : 9.828955e-13
uncertainty floor        : 1.965791e-11


In [8]:
# ============================================================
# Cell 8 — PAIRWISE LOCAL INTERACTION LIBRARY (BLIND)
# ============================================================
#
# This is the new representation layer between preprocessing
# and TIDES Steps 2/3.
#
# The incidence matrix is used only to orient/gather endpoints.
# Dynamics are represented by edge-local endpoint atoms
#
#   Gamma_l(x_i,x_j) = [g_i, g_j].
#
# Completely unknown dynamics -> polynomial mode.
# The true theta / active snapshots / edge weights are NOT used.
# ============================================================

import importlib
import pairwise_local_interaction_library

importlib.reload(pairwise_local_interaction_library)

from pairwise_local_interaction_library import (
    build_pairwise_local_interaction_library,
)

M = len(CANDIDATE_EDGES)
K = len(detected_transition_indices)

# Standard oriented incidence: edge (i,j), i<j, is -e_i + e_j.
D = np.zeros((N, M), dtype=float)
for m, (i, j) in enumerate(CANDIDATE_EDGES):
    D[i - 1, m] = -1.0
    D[j - 1, m] = +1.0

LOCAL_LIBRARY = build_pairwise_local_interaction_library(
    X_MID,
    D,
    mode="polynomial",
    degree=5,
    normalization="none",
)

EDGE_PSI = LOCAL_LIBRARY.endpoint_features
L = LOCAL_LIBRARY.n_features
print("EDGE_PSI shape :", EDGE_PSI.shape)
print("feature labels :", LOCAL_LIBRARY.feature_labels)
print("components     :", LOCAL_LIBRARY.component_labels)

# --------------------------------------------------------------------------
# ORACLE INTERACTION LAW — VALIDATION ONLY
# --------------------------------------------------------------------------

TRUE_THETA = np.zeros(LOCAL_LIBRARY.n_features, dtype=float)

label_to_idx = {
    label: i
    for i, label in enumerate(LOCAL_LIBRARY.feature_labels)
}

TRUE_THETA[label_to_idx["neighbor"]] = 1.0
TRUE_THETA[label_to_idx["neighbor^2"]] = 0.18
TRUE_THETA[label_to_idx["self*neighbor"]] = -0.62

print("oracle theta   :", TRUE_THETA)

print("=" * 90)
print("Cell 8 — PAIRWISE LOCAL INTERACTION LIBRARY")
print("=" * 90)
print("candidate edges M       :", M)
print("incidence rank          :", np.linalg.matrix_rank(D))
print("library mode            :", LOCAL_LIBRARY.mode)
print("representation          :", LOCAL_LIBRARY.feature_representation)
print("feature labels          :", LOCAL_LIBRARY.feature_labels)
print("component labels        :", LOCAL_LIBRARY.component_labels)
print("endpoint states shape   :", LOCAL_LIBRARY.endpoint_states.shape)
print("local (c,d) shape       :", LOCAL_LIBRARY.local_states.shape)
print("endpoint feature shape  :", EDGE_PSI.shape)
print("features per edge L     :", L)
print("candidate change groups :", K * M)

# The default polynomial sector excludes pure own-state monomials,
# preventing the edge-self gauge in the general endpoint-output model.
assert D.shape == (8, 28)
assert LOCAL_LIBRARY.mode == "polynomial"
assert LOCAL_LIBRARY.feature_representation == "endpoint_pairwise_irreducible_polynomial"
assert LOCAL_LIBRARY.feature_labels == (
    "neighbor",
    "neighbor^2",
    "self*neighbor",
)
assert EDGE_PSI.shape == (708, 28, 3, 2)
assert LOCAL_LIBRARY.endpoint_states.shape == (708, 28, 2)
assert LOCAL_LIBRARY.local_states.shape == (708, 28, 2)
assert LOCAL_LIBRARY.metadata["one_body_sector_excluded"] is True

print("\noracle information used : NO")
print("LOCAL INTERACTION LIBRARY CONSTRUCTION: PASS")


Cell 8 — PAIRWISE LOCAL INTERACTION LIBRARY
candidate edges M       : 28
incidence rank          : 7
library mode            : polynomial
representation          : endpoint_pairwise_irreducible_polynomial
feature labels          : ('neighbor', 'neighbor^2', 'self*neighbor')
component labels        : ('poly:neighbor', 'poly:neighbor^2', 'poly:self*neighbor')
endpoint states shape   : (708, 28, 2)
local (c,d) shape       : (708, 28, 2)
endpoint feature shape  : (708, 28, 3, 2)
features per edge L     : 3
candidate change groups : 140

oracle information used : NO
LOCAL INTERACTION LIBRARY CONSTRUCTION: PASS


In [9]:
# ============================================================
# Cell 9 — TIDES STEP 2: DENSE vs KKT-WORKING-SET REGRESSION
# ============================================================
# Both backends solve the SAME grouped BPDN + prefix-floor task.
# The scalable backend now uses:
#
#   batched feasibility seed
#   -> restricted grouped BPDN
#   -> global dual/KKT audit
#   -> reactivation if needed.
#
# The endpoint-output library tensor is passed unchanged.
# ============================================================

import importlib
import step2_change_structure

importlib.reload(step2_change_structure)

from step2_change_structure import (
    infer_change_structure_from_observations,
)

DETECTED_TRANSITION_INDICES = np.asarray(
    s1_blind.transition_indices,
    dtype=int,
)
DETECTED_TRANSITION_TIMES = t[DETECTED_TRANSITION_INDICES]

COMMON_STEP2 = dict(
    Y=V_MID,
    D=D,
    edge_features=EDGE_PSI,
    stage_of_sample=OBS_STAGE,
    hypothesis="varying_structure",
    transition_indices=DETECTED_TRANSITION_INDICES,
    transition_times=DETECTED_TRANSITION_TIMES,
    edge_labels=CANDIDATE_EDGES,
    uncertainty_floor=UNCERTAINTY_FLOOR,
    solver_method="group_bpdn_prefix",
    refit_method="dense_lstsq",
    require_solver_convergence=True,
    require_floor_reached=True,
    return_design=False,
)

print("=" * 90)
print("STEP 2 — DENSE REFERENCE")
print("=" * 90)

STEP2_DENSE = infer_change_structure_from_observations(
    **COMMON_STEP2,
    backend="dense",
    solver_kwargs={
        "max_iter": 10000,
        "tol": 1e-8,
        "check_every": 50,
        "verbose": True,
    },
)

print("\n" + "=" * 90)
print("STEP 2 — SCALABLE KKT-WORKING-SET")
print("=" * 90)

STEP2_SCALABLE = infer_change_structure_from_observations(
    **COMMON_STEP2,
    backend="scalable",
    scalable_solver="working_set",
    solver_kwargs={
        "max_iter": 10000,
        "tol": 1e-8,
        "check_every": 50,
        "seed_batch_size": 8,
        "seed_growth_factor": 1.0,
        "kkt_tol": 1e-7,
        "verbose": True,
    },
)

dense_groups = tuple(STEP2_DENSE.selected_groups)
scalable_groups = tuple(STEP2_SCALABLE.selected_groups)
dense_set = set(dense_groups)
scalable_set = set(scalable_groups)
same_support = dense_set == scalable_set

dense_prefix = STEP2_DENSE.selection_result.selected_prefix_size
scalable_prefix = STEP2_SCALABLE.selection_result.selected_prefix_size

print("\n" + "=" * 90)
print("Cell 9 — DENSE / SCALABLE REGRESSION")
print("=" * 90)
print("feature representation      :", STEP2_SCALABLE.metadata["feature_representation"])
print("output generalization active:", STEP2_SCALABLE.metadata["output_generalization_active"])
print("dense selected groups       :", len(dense_groups))
print("scalable selected groups    :", len(scalable_groups))
print("dense selected prefix       :", dense_prefix)
print("scalable selected prefix    :", scalable_prefix)
print("dense final residual        :", f"{STEP2_DENSE.relative_residual:.6e}")
print("scalable final residual     :", f"{STEP2_SCALABLE.relative_residual:.6e}")
print("uncertainty floor           :", f"{UNCERTAINTY_FLOOR:.6e}")
print("same final support          :", same_support)
print("seed groups                 :", STEP2_SCALABLE.metadata["seed_group_count"])
print("working-set groups          :", STEP2_SCALABLE.metadata["working_set_group_count"])
print("KKT reactivations           :", STEP2_SCALABLE.metadata["total_reactivations"])
print("KKT audits                  :", STEP2_SCALABLE.metadata["kkt_audits"])
print("max global dual ratio       :", f"{STEP2_SCALABLE.metadata['max_global_dual_ratio']:.12e}")
print("global dual feasible        :", STEP2_SCALABLE.metadata["global_dual_feasible"])
print("global duality gap          :", f"{STEP2_SCALABLE.metadata['global_duality_gap']:.6e}")

print("\n" + "-" * 90)
print("TRANSITION-WISE SUPPORTS")
print("-" * 90)
for dense_c, scalable_c in zip(
    STEP2_DENSE.constraints,
    STEP2_SCALABLE.constraints,
):
    print(
        f"transition {dense_c.transition_ordinal + 1} "
        f"@ t={dense_c.transition_time:.6f}"
    )
    print("  dense    :", dense_c.support_labels)
    print("  scalable :", scalable_c.support_labels)

# Backend-to-backend regression (no truth).
assert STEP2_DENSE.metadata["computational_backend_used"] == "dense"
assert STEP2_SCALABLE.metadata["computational_backend_used"] == "scalable"
assert STEP2_DENSE.metadata["global_change_design_materialized"]
assert not STEP2_SCALABLE.metadata["global_change_design_materialized"]
assert STEP2_DENSE.relative_residual <= UNCERTAINTY_FLOOR
assert STEP2_SCALABLE.relative_residual <= UNCERTAINTY_FLOOR
assert STEP2_SCALABLE.metadata["global_dual_feasible"]
assert STEP2_SCALABLE.metadata["max_global_dual_ratio"] <= 1.0 + 1e-7
assert same_support
assert dense_prefix == scalable_prefix

# ------------------------------------------------------------
# ORACLE VALIDATION ONLY: exact temporal-edge support
# ------------------------------------------------------------
TRUE_CHANGE_GROUPS = {
    (k, EDGE_INDEX_1B[edge])
    for k, support in enumerate(transition_supports_true)
    for edge in support
}

support_exact = scalable_set == TRUE_CHANGE_GROUPS
TP = len(scalable_set & TRUE_CHANGE_GROUPS)
FP = len(scalable_set - TRUE_CHANGE_GROUPS)
FN = len(TRUE_CHANGE_GROUPS - scalable_set)

print("\n" + "-" * 90)
print("STEP-2 ORACLE VALIDATION ONLY")
print("-" * 90)
print("true groups                 :", len(TRUE_CHANGE_GROUPS))
print("selected groups             :", len(scalable_set))
print("TP / FP / FN                :", TP, "/", FP, "/", FN)
print("support exact               :", support_exact)

assert len(TRUE_CHANGE_GROUPS) == 20
assert support_exact

STEP2_RESULT = STEP2_SCALABLE

print("\nTIDES STEP 2 DENSE/SCALABLE + SUPPORT REGRESSION: PASS")


STEP 2 — DENSE REFERENCE
Starting grouped basis-pursuit denoising...
  samples=5664 | coefficients=420 | groups=140
  residual radius=8.732e-11 | relative target=1.966e-11
  backend=dense-svd Douglas-Rachford | rank=241/420 | DR step=1.182e-01
  support selection=disabled here; solver returns group ranking only
  [iter      1] fixed-point=7.824e-01 | rel-res=1.966e-11
  [iter    500] fixed-point=4.202e-05 | rel-res=1.964e-11
  [iter   1000] fixed-point=1.215e-05 | rel-res=1.964e-11
  [iter   1500] fixed-point=5.161e-06 | rel-res=1.964e-11
  [iter   2000] fixed-point=2.402e-06 | rel-res=1.964e-11
  [iter   2500] fixed-point=1.180e-06 | rel-res=1.964e-11
  [iter   3000] fixed-point=6.011e-07 | rel-res=1.964e-11
  [iter   3500] fixed-point=3.146e-07 | rel-res=1.964e-11
  [iter   4000] fixed-point=1.688e-07 | rel-res=1.964e-11
  [iter   4500] fixed-point=9.301e-08 | rel-res=1.964e-11
  [iter   5000] fixed-point=5.282e-08 | rel-res=1.964e-11
  [iter   5500] fixed-point=3.097e-08 | rel-res=1

In [10]:
# ============================================================
# Cell 10 — TIDES STEP 3 (BLIND):
#           selected-support vector-field reconstruction
# ============================================================

import importlib
import solvers_linear_regression
import step3_vector_field

importlib.reload(solvers_linear_regression)
importlib.reload(step3_vector_field)

from step3_vector_field import (
    reconstruct_vector_field_from_observations,
)

STEP3_RESULT = reconstruct_vector_field_from_observations(
    Y=V_MID,
    D=D,
    edge_features=EDGE_PSI,
    stage_of_sample=OBS_STAGE,
    change_constraints_or_supports=STEP2_RESULT,
    solver_method="dense_lstsq",
    solver_kwargs={
        "rcond": None,
        "compute_raw_svd_diagnostics": True,
        "verbose": True,
    },
    design_mode="dense",
    return_design=True,
)

design = STEP3_RESULT.design
solver = STEP3_RESULT.solver_result
selected_change_groups = sum(len(c.support) for c in STEP2_RESULT.constraints)
expected_parameter_count = M * L + selected_change_groups * L

print("\n" + "=" * 90)
print("Cell 10 — TIDES STEP 3 (BLIND): RESULT")
print("=" * 90)
print("feature representation     :", STEP3_RESULT.metadata["feature_representation"])
print("output generalization      :", STEP3_RESULT.metadata["output_generalization_active"])
print("preprocessed observations  :", STEP3_RESULT.n_preprocessed_observations)
print("scalar observations        :", STEP3_RESULT.n_observations)
print("selected change groups     :", selected_change_groups)
print("anchor parameters          :", M * L)
print("change parameters          :", selected_change_groups * L)
print("total parameters           :", STEP3_RESULT.parameter_count)
print("design shape               :", design.matrix.shape)
print("solver method              :", solver.method)
print("rank                       :", f"{STEP3_RESULT.rank}/{STEP3_RESULT.parameter_count}")
print("identifiable               :", STEP3_RESULT.identifiable)
print("condition number (scaled)  :", f"{STEP3_RESULT.condition_number_scaled:.6e}")
print("condition number (raw)     :", f"{STEP3_RESULT.condition_number_raw:.6e}")
print("relative residual          :", f"{STEP3_RESULT.relative_residual:.6e}")
print("normal-equation residual   :", f"{solver.normal_equation_relative_residual:.6e}")
print("uncertainty floor          :", f"{UNCERTAINTY_FLOOR:.6e}")
print("below data resolution      :", STEP3_RESULT.relative_residual <= UNCERTAINTY_FLOOR)
print("B_anchor shape             :", STEP3_RESULT.B_anchor.shape)
print("number of Delta B blocks   :", len(STEP3_RESULT.delta_B))
print("B_stages shape             :", STEP3_RESULT.B_stages.shape)
print("oracle information used    : NO")

assert STEP3_RESULT.parameter_count == expected_parameter_count
assert design.matrix.shape == (len(V_MID) * N, expected_parameter_count)
assert STEP3_RESULT.B_anchor.shape == (M, L)
assert len(STEP3_RESULT.delta_B) == len(STEP2_RESULT.constraints)
assert all(dB.shape == (M, L) for dB in STEP3_RESULT.delta_B)
assert STEP3_RESULT.B_stages.shape == (
    len(STEP2_RESULT.constraints) + 1,
    M,
    L,
)
assert np.all(np.isfinite(STEP3_RESULT.B_anchor))
assert np.all(np.isfinite(STEP3_RESULT.B_stages))
assert solver.converged
if STEP3_RESULT.rank is not None:
    assert STEP3_RESULT.rank == STEP3_RESULT.parameter_count
assert STEP3_RESULT.relative_residual <= UNCERTAINTY_FLOOR

print("\nTIDES STEP 3 RECONSTRUCTION: PASS")


Starting linear least-squares solve...
  samples=5664 | features=144 | outputs=1 | method=dense_lstsq
Linear least-squares solve complete.
  relative residual=1.423e-13 | normal-eq residual=7.564e-03
  rank=144/144 | cond(scaled)=3.159e+06 | cond(raw)=8.070e+06

Cell 10 — TIDES STEP 3 (BLIND): RESULT
feature representation     : endpoint_pairwise
output generalization      : True
preprocessed observations  : 708
scalar observations        : 5664
selected change groups     : 20
anchor parameters          : 84
change parameters          : 60
total parameters           : 144
design shape               : (5664, 144)
solver method              : dense_lstsq
rank                       : 144/144
identifiable               : True
condition number (scaled)  : 3.159262e+06
condition number (raw)     : 8.070032e+06
relative residual          : 1.423470e-13
normal-equation residual   : 7.563744e-03
uncertainty floor          : 1.965791e-11
below data resolution      : True
B_anchor shape          

In [11]:
# ============================================================
# Cell 10A — TIDES STEP 3 VALIDATION:
#            stage-wise coefficient recovery
# ORACLE / BENCHMARK VALIDATION ONLY
# ============================================================

B_TRUE_STAGES = np.zeros(
    (N_STAGES, M, L),
    dtype=float,
)

for r, snapshot in enumerate(SNAPSHOTS):
    for edge in snapshot:
        m = EDGE_INDEX_1B[edge]
        B_TRUE_STAGES[r, m, :] = EDGE_WEIGHT[edge] * PAIR_LAW_TRUE

DELTA_B_TRUE = np.diff(B_TRUE_STAGES, axis=0)
B_HAT_STAGES = STEP3_RESULT.B_stages
DELTA_B_HAT = np.stack(STEP3_RESULT.delta_B, axis=0)

anchor_relative_error = (
    np.linalg.norm(STEP3_RESULT.B_anchor - B_TRUE_STAGES[0])
    / np.linalg.norm(B_TRUE_STAGES[0])
)
stage_relative_error = (
    np.linalg.norm(B_HAT_STAGES - B_TRUE_STAGES)
    / np.linalg.norm(B_TRUE_STAGES)
)
delta_relative_error = (
    np.linalg.norm(DELTA_B_HAT - DELTA_B_TRUE)
    / np.linalg.norm(DELTA_B_TRUE)
)
max_abs_stage_error = np.max(np.abs(B_HAT_STAGES - B_TRUE_STAGES))
max_abs_delta_error = np.max(np.abs(DELTA_B_HAT - DELTA_B_TRUE))

per_stage_errors = []
for r in range(N_STAGES):
    truth_norm = np.linalg.norm(B_TRUE_STAGES[r])
    per_stage_errors.append(
        float(
            np.linalg.norm(B_HAT_STAGES[r] - B_TRUE_STAGES[r])
            / max(truth_norm, np.finfo(float).tiny)
        )
    )

per_transition_errors = []
for k in range(N_STAGES - 1):
    truth_norm = np.linalg.norm(DELTA_B_TRUE[k])
    per_transition_errors.append(
        float(
            np.linalg.norm(DELTA_B_HAT[k] - DELTA_B_TRUE[k])
            / max(truth_norm, np.finfo(float).tiny)
        )
    )

true_zero_mask = np.isclose(B_TRUE_STAGES, 0.0, atol=0.0)
max_inactive_coefficient = np.max(np.abs(B_HAT_STAGES[true_zero_mask]))

print("=" * 90)
print("Cell 10A — TIDES STEP 3 VALIDATION")
print("=" * 90)
print("true theta coordinates      :", PAIR_LAW_LABELS_TRUE)
print("anchor relative error       :", f"{anchor_relative_error:.6e}")
print("all-stage relative error    :", f"{stage_relative_error:.6e}")
print("all-delta relative error    :", f"{delta_relative_error:.6e}")
print("max |B_hat - B_true|        :", f"{max_abs_stage_error:.6e}")
print("max |dB_hat - dB_true|      :", f"{max_abs_delta_error:.6e}")
print("max inactive coefficient    :", f"{max_inactive_coefficient:.6e}")

for r, err in enumerate(per_stage_errors):
    print(f"stage {r + 1} relative error      : {err:.6e}")
for k, err in enumerate(per_transition_errors):
    print(f"transition {k + 1} delta error   : {err:.6e}")

VALIDATION_TOL = 1e-7
assert anchor_relative_error < VALIDATION_TOL
assert stage_relative_error < VALIDATION_TOL
assert delta_relative_error < VALIDATION_TOL

print("\nORACLE INFORMATION USED: YES — VALIDATION ONLY")
print("TIDES STEP 3 COEFFICIENT RECOVERY: PASS")


Cell 10A — TIDES STEP 3 VALIDATION
true theta coordinates      : ('neighbor', 'neighbor^2', 'self*neighbor')
anchor relative error       : 2.704033e-09
all-stage relative error    : 2.955099e-09
all-delta relative error    : 2.393000e-09
max |B_hat - B_true|        : 5.167437e-09
max |dB_hat - dB_true|      : 5.387613e-09
max inactive coefficient    : 4.682283e-09
stage 1 relative error      : 2.704033e-09
stage 2 relative error      : 2.804570e-09
stage 3 relative error      : 2.837192e-09
stage 4 relative error      : 2.702150e-09
stage 5 relative error      : 3.062585e-09
stage 6 relative error      : 3.567538e-09
transition 1 delta error   : 1.669262e-09
transition 2 delta error   : 8.734552e-10
transition 3 delta error   : 2.805454e-09
transition 4 delta error   : 2.032310e-09
transition 5 delta error   : 3.553254e-09

ORACLE INFORMATION USED: YES — VALIDATION ONLY
TIDES STEP 3 COEFFICIENT RECOVERY: PASS


In [12]:
# ============================================================
# Cell 11 — TIDES STEP 4 (BLIND):
#           shared-law source decomposition
# ============================================================

import importlib
import step4_source_decomposition

importlib.reload(step4_source_decomposition)

from step4_source_decomposition import (
    decompose_vector_field_sources,
)

STEP4_RESULT = decompose_vector_field_sources(
    STEP3_RESULT,
    hypothesis="shared_interaction_law",
    normalization="reference_component",
    reference_component=0,
    component_labels=LOCAL_LIBRARY.component_labels,
)

print("=" * 90)
print("Cell 11 — TIDES STEP 4 (BLIND): RESULT")
print("=" * 90)
print("component labels           :", LOCAL_LIBRARY.component_labels)
print("B_stages shape             :", STEP3_RESULT.B_stages.shape)
print("W_stages shape             :", STEP4_RESULT.W_stages.shape)
print("theta shape                :", STEP4_RESULT.theta.shape)
print("inferred theta             :", np.array2string(STEP4_RESULT.theta, precision=12))
print("unit-gauge theta           :", np.array2string(STEP4_RESULT.theta_unit, precision=12))
print("gauge normalization        :", STEP4_RESULT.normalization)
print("gauge component            :", STEP4_RESULT.gauge_component)

print("\n" + "-" * 90)
print("SHARED-LAW DIAGNOSTICS")
print("-" * 90)
print("singular values            :", np.array2string(STEP4_RESULT.singular_values, precision=12))
print("numerical SVD rank         :", STEP4_RESULT.numerical_rank)
print("rank-1 energy fraction     :", f"{STEP4_RESULT.rank1_energy_fraction:.16f}")
print("s2 / s1                    :", f"{STEP4_RESULT.second_to_first_singular_ratio:.6e}")
print("spectral gap s1 / s2       :", f"{STEP4_RESULT.spectral_gap:.6e}")
print("rank-1 relative residual   :", f"{STEP4_RESULT.relative_residual:.6e}")

B_FROM_FACTORS = (
    STEP4_RESULT.W_stages[..., None]
    * STEP4_RESULT.theta[None, None, :]
)
factorization_consistency = (
    np.linalg.norm(B_FROM_FACTORS - STEP4_RESULT.B_reconstructed)
    / max(np.linalg.norm(STEP4_RESULT.B_reconstructed), np.finfo(float).tiny)
)

print("\nfactor reconstruction err  :", f"{factorization_consistency:.6e}")
print("theta[0] after gauge fix   :", f"{STEP4_RESULT.theta[0]:.12f}")
print("oracle information used    : NO")

R, M_, L_ = STEP3_RESULT.B_stages.shape
assert STEP4_RESULT.W_stages.shape == (R, M_)
assert STEP4_RESULT.theta.shape == (L_,)
assert STEP4_RESULT.B_reconstructed.shape == (R, M_, L_)
assert np.all(np.isfinite(STEP4_RESULT.theta))
assert np.all(np.isfinite(STEP4_RESULT.W_stages))
assert np.isclose(STEP4_RESULT.theta[0], 1.0, rtol=1e-12, atol=1e-12)
assert factorization_consistency < 1e-12

print("\nTIDES STEP 4 SOURCE DECOMPOSITION: PASS")


Cell 11 — TIDES STEP 4 (BLIND): RESULT
component labels           : ('poly:neighbor', 'poly:neighbor^2', 'poly:self*neighbor')
B_stages shape             : (6, 28, 3)
W_stages shape             : (6, 28)
theta shape                : (3,)
inferred theta             : [ 1.              0.180000000398 -0.620000000154]
unit-gauge theta           : [ 0.840128515788  0.151223133176 -0.520879679918]
gauge normalization        : reference_component
gauge component            : 0

------------------------------------------------------------------------------------------
SHARED-LAW DIAGNOSTICS
------------------------------------------------------------------------------------------
singular values            : [8.897076593466e+00 1.974620246622e-08 1.324840769249e-08]
numerical SVD rank         : 3
rank-1 energy fraction     : 1.0000000000000000
s2 / s1                    : 2.219403e-09
spectral gap s1 / s2       : 4.505715e+08
rank-1 relative residual   : 2.672657e-09

factor reconstruction er

In [ ]:
print("\n" + "=" * 90)
print("STEP 4 — ORACLE VALIDATION ONLY")
print("=" * 90)

theta_hat = np.asarray(STEP4_RESULT.theta, dtype=float)
theta_true = np.asarray(TRUE_THETA, dtype=float)

theta_rel_err = (
    np.linalg.norm(theta_hat - theta_true)
    / np.linalg.norm(theta_true)
)

high_order = theta_hat[3:]

print("true theta               :", theta_true)
print("inferred theta           :", theta_hat)
print("theta relative error      :", f"{theta_rel_err:.6e}")
print("higher-order L2 norm      :", f"{np.linalg.norm(high_order):.6e}")
print("higher-order max abs      :", f"{np.max(np.abs(high_order)):.6e}")

In [13]:
# ============================================================
# Cell 11A — TIDES STEP 4 VALIDATION:
#            microscopic source + shared-law recovery
# ORACLE / BENCHMARK VALIDATION ONLY
# ============================================================

W_TRUE_STAGES = np.zeros((N_STAGES, M), dtype=float)
for r, snapshot in enumerate(SNAPSHOTS):
    for edge in snapshot:
        W_TRUE_STAGES[r, EDGE_INDEX_1B[edge]] = EDGE_WEIGHT[edge]

THETA_TRUE = np.asarray(PAIR_LAW_TRUE, dtype=float)
THETA_HAT = np.asarray(STEP4_RESULT.theta, dtype=float)
W_HAT_STAGES = np.asarray(STEP4_RESULT.W_stages, dtype=float)

theta_absolute_error = np.linalg.norm(THETA_HAT - THETA_TRUE)
theta_relative_error = theta_absolute_error / np.linalg.norm(THETA_TRUE)
theta_max_abs_error = np.max(np.abs(THETA_HAT - THETA_TRUE))

W_relative_error = (
    np.linalg.norm(W_HAT_STAGES - W_TRUE_STAGES)
    / np.linalg.norm(W_TRUE_STAGES)
)
W_max_abs_error = np.max(np.abs(W_HAT_STAGES - W_TRUE_STAGES))

inactive_mask = np.isclose(W_TRUE_STAGES, 0.0, atol=0.0)
active_mask = ~inactive_mask
max_inactive_W = np.max(np.abs(W_HAT_STAGES[inactive_mask]))
active_W_relative_error = (
    np.linalg.norm(W_HAT_STAGES[active_mask] - W_TRUE_STAGES[active_mask])
    / np.linalg.norm(W_TRUE_STAGES[active_mask])
)

B_MICRO_HAT = W_HAT_STAGES[..., None] * THETA_HAT[None, None, :]
B_TRUE_FROM_SOURCES = W_TRUE_STAGES[..., None] * THETA_TRUE[None, None, :]
microscopic_B_relative_error = (
    np.linalg.norm(B_MICRO_HAT - B_TRUE_FROM_SOURCES)
    / np.linalg.norm(B_TRUE_FROM_SOURCES)
)

print("=" * 90)
print("Cell 11A — TIDES STEP 4 VALIDATION")
print("=" * 90)
print("theta labels              :", PAIR_LAW_LABELS_TRUE)
print("theta true                :", np.array2string(THETA_TRUE, precision=12))
print("theta inferred            :", np.array2string(THETA_HAT, precision=12))
print("theta absolute error      :", f"{theta_absolute_error:.6e}")
print("theta relative error      :", f"{theta_relative_error:.6e}")
print("theta max abs error       :", f"{theta_max_abs_error:.6e}")
print("all-W relative error      :", f"{W_relative_error:.6e}")
print("active-W relative error   :", f"{active_W_relative_error:.6e}")
print("max |W_hat - W_true|      :", f"{W_max_abs_error:.6e}")
print("max inactive W            :", f"{max_inactive_W:.6e}")
print("microscopic B rel. error  :", f"{microscopic_B_relative_error:.6e}")

VALIDATION_TOL = 1e-7
assert theta_relative_error < VALIDATION_TOL
assert W_relative_error < VALIDATION_TOL
assert microscopic_B_relative_error < VALIDATION_TOL

print("\nORACLE INFORMATION USED: YES — VALIDATION ONLY")
print("TIDES STEP 4 SOURCE RECOVERY: PASS")


Cell 11A — TIDES STEP 4 VALIDATION
theta labels              : ('neighbor', 'neighbor^2', 'self*neighbor')
theta true                : [ 1.    0.18 -0.62]
theta inferred            : [ 1.              0.180000000398 -0.620000000154]
theta absolute error      : 4.266271e-10
theta relative error      : 3.584216e-10
theta max abs error       : 3.980294e-10
all-W relative error      : 1.200239e-09
active-W relative error   : 8.340668e-10
max |W_hat - W_true|      : 1.931693e-09
max inactive W            : 1.739221e-09
microscopic B rel. error  : 1.260761e-09

ORACLE INFORMATION USED: YES — VALIDATION ONLY
TIDES STEP 4 SOURCE RECOVERY: PASS


In [14]:
# ============================================================
# Cell 12 — Canonical N=8 benchmark summary
# ============================================================

print("=" * 90)
print("NEW-FRAMEWORK N=8 BENCHMARK SUMMARY")
print("=" * 90)
print("Step 1 exact transitions       :", blind_exact)
print("Library representation         :", LOCAL_LIBRARY.feature_representation)
print("Endpoint-output tensor         :", EDGE_PSI.shape)
print("Step 2 exact support           :", support_exact)
print("Step 2 dense/scalable agree    :", same_support)
print("Step 2 global KKT feasible     :", STEP2_SCALABLE.metadata["global_dual_feasible"])
print("Step 3 identifiable            :", STEP3_RESULT.identifiable)
print("Step 3 B relative error        :", f"{stage_relative_error:.6e}")
print("Step 4 theta relative error    :", f"{theta_relative_error:.6e}")
print("Step 4 rank-1 residual         :", f"{STEP4_RESULT.relative_residual:.6e}")
print("Non-conservative sum change    :", f"{net_sum_change:.6e}")

assert blind_exact
assert support_exact
assert same_support
assert STEP2_SCALABLE.metadata["global_dual_feasible"]
assert STEP3_RESULT.identifiable is True
assert stage_relative_error < 1e-7
assert theta_relative_error < 1e-7
assert abs(net_sum_change) > 1e-3

print("\nALL NEW-FRAMEWORK N=8 REGRESSION CHECKS: PASS")


NEW-FRAMEWORK N=8 BENCHMARK SUMMARY
Step 1 exact transitions       : True
Library representation         : endpoint_pairwise_irreducible_polynomial
Endpoint-output tensor         : (708, 28, 3, 2)
Step 2 exact support           : True
Step 2 dense/scalable agree    : True
Step 2 global KKT feasible     : True
Step 3 identifiable            : True
Step 3 B relative error        : 2.955099e-09
Step 4 theta relative error    : 3.584216e-10
Step 4 rank-1 residual         : 2.672657e-09
Non-conservative sum change    : 4.504142e-01

ALL NEW-FRAMEWORK N=8 REGRESSION CHECKS: PASS
